# 09 · TotalSpineSeg en Kaggle para RSNA (cuaderno auxiliar)

Este cuaderno segmenta con TotalSpineSeg las series de RSNA de una de las dos listas del cuaderno 08. Las cabeceras DICOM de RSNA están anonimizadas y, al leerlas con GDCM, la imagen queda con espaciado de 1 × 1 × 1 mm y origen en (0, 0, 0), aunque las cabeceras traen la posición, la orientación y el tamaño del píxel de cada imagen. Por eso aquí los vóxeles se leen igual que en el 08, y se comprueba con la huella del 08 que son los mismos, pero el espaciado, el origen y la dirección se toman de las cabeceras. Las series de Mendeley no tienen ese problema y se segmentan con el cuaderno auxiliar del 08.

**Antes de correrlo, en Kaggle:**

1. *Settings → Accelerator*: **GPU T4 x2**. *Settings → Internet*: **On** (para instalar TotalSpineSeg y bajar sus pesos).
2. *Add Input*: el dataset privado **prl-08-entradas** (lo sube el cuaderno 08) y la competición **RSNA 2024 Lumbar Spine Degenerative Classification** (la cuenta tiene que haber aceptado sus reglas).
3. En la celda 2, `LISTA` es `"A"` en una cuenta y `"B"` en la otra, e `INICIO` es 1.
4. *Save Version → Save & Run All (Commit)*. No hace falta probar antes en modo interactivo: la celda 4 convierte una serie de prueba y detiene la versión si los vóxeles no son los del 08 o si la geometría no quedó bien, y la celda 5 se detiene si el primer lote no da ninguna máscara o si alguna máscara sale con espaciado de 1 mm y origen en cero. En esos casos la versión se corta a los pocos minutos y casi no gasta cuota de GPU.

Cada lista tiene unas 1.975 series de RSNA, y con dos GPU la corrida debería tomar del orden de 5 horas. La celda de segmentación deja de tomar lotes nuevos antes de las 12 horas de Kaggle (`HORAS_MAXIMAS`) y al final imprime el `INICIO` de la siguiente versión, por si hiciera falta. Cada máscara se agrega al zip de salida apenas se produce. La salida se guarda en `content/datos/segmentacion_tss/rsna/` de Drive, que es donde la lee el cuaderno 09.

Las máscaras de RSNA se quedan en la salida privada del cuaderno y después en la carpeta privada de Drive; no se publican, por los términos de RSNA.

In [ ]:
# 1. Entorno aparte para TotalSpineSeg (igual que en el cuaderno 05) y lectura de cabeceras DICOM
import subprocess, sys

def correr(cmd, **kw):
    "Corre un comando y, si falla, muestra su salida de error antes de detenerse."
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise RuntimeError(f"Falló: {cmd if isinstance(cmd, str) else ' '.join(cmd)}")
    return r

correr("pip -q install virtualenv SimpleITK pydicom", shell=True)
correr("rm -rf /kaggle/tmp/venv_tss && virtualenv -q --system-site-packages /kaggle/tmp/venv_tss", shell=True)
correr(["/kaggle/tmp/venv_tss/bin/pip", "-q", "install", "totalspineseg==20260807", "nnunetv2==2.6.2", "kornia==0.8.2"])
r = correr(["/kaggle/tmp/venv_tss/bin/python", "-c",
            "import torch; from importlib.metadata import version as v; "
            "print(torch.__version__, torch.cuda.device_count(), v('totalspineseg'))"])
version_torch, gpus_tss, version_tss = r.stdout.split()
print(f"torch {version_torch} | GPUs: {gpus_tss} | totalspineseg {version_tss}")
assert int(gpus_tss) >= 1, "TotalSpineSeg no ve ninguna GPU: revisa Settings → Accelerator antes de seguir"
assert version_tss == "20260807", "Versión de TotalSpineSeg distinta de la fijada"

In [ ]:
# 2. Parámetros y rutas
LISTA = "A"              # "A" en una cuenta, "B" en la otra
INICIO = 1               # posición de la lista desde la que se empieza esta versión
HORAS_MAXIMAS = 11.0     # se dejan de tomar lotes nuevos pasado este tiempo (Kaggle corta a las 12 h)
TAM_LOTE = 20            # series por llamada a TotalSpineSeg: el modelo se carga una vez por lote
FIN = "" if INICIO == 1 else f"_desde_{INICIO:05d}"     # va en el nombre de los archivos de salida de cada versión

import os, time, json, shutil, hashlib, datetime, zipfile, importlib.util
from pathlib import Path
import numpy as np
import pandas as pd
import SimpleITK as sitk
import pydicom

ENTRADA = Path("/kaggle/input")
TRABAJO = Path("/kaggle/working")
TMP = Path("/kaggle/tmp")
VENV_TSS = TMP / "venv_tss"
DIR_TSS_DATOS = TMP / "tss_datos"
DIR_MASCARAS = TMP / "mascaras"                     # paso intermedio; cada máscara se agrega enseguida al zip de salida
for d in (DIR_TSS_DATOS, DIR_MASCARAS):
    d.mkdir(parents=True, exist_ok=True)
CARPETAS_DE_IMAGENES = {"train_images", "test_images"}   # no se recorren al buscar archivos: tienen ~150.000 imágenes

def buscar(patron):
    "Archivos de las entradas que cumplen el patrón, sin entrar en las carpetas de imágenes de la competición."
    hallados = []
    for raiz, dirs, archivos in os.walk(ENTRADA):
        dirs[:] = [d for d in dirs if d not in CARPETAS_DE_IMAGENES]
        hallados += [Path(raiz) / a for a in archivos if Path(a).match(patron)]
    return sorted(hallados)

def unico(patron):
    hallados = buscar(patron)
    assert hallados, f"No se encontró {patron} en /kaggle/input: revisa las entradas del cuaderno"
    return hallados[0]

RUTA_LISTA = unico(f"lista_{LISTA}.csv")
RUTA_CONVERSION = unico("conversion08.py")
candidatas = [p for nivel in ("*", "*/*", "*/*/*", "*/*/*/*") for p in ENTRADA.glob(f"{nivel}/train_images") if p.is_dir()]
assert candidatas, "Falta la competición de RSNA entre las entradas"
DIR_RSNA = candidatas[0]

spec = importlib.util.spec_from_file_location("conversion08", RUTA_CONVERSION)
conversion08 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(conversion08)

lista_completa = pd.read_csv(RUTA_LISTA, dtype={"estudio": str, "serie_clave": str})
lista = lista_completa[lista_completa["conjunto"] == "rsna"].copy()
print(f"Lista {LISTA}: {len(lista_completa)} series; esta corrida segmenta solo las {len(lista)} de RSNA")
print("Lista en:", RUTA_LISTA.parent, "| imágenes de RSNA en:", DIR_RSNA)
faltan_carpetas = [r.id_serie for r in lista.head(50).itertuples() if not (DIR_RSNA / r.estudio / r.serie_clave).is_dir()]
assert not faltan_carpetas, f"{len(faltan_carpetas)} de las primeras 50 series no están en la competición"

In [ ]:
# 3. Pesos de TotalSpineSeg de la release fijada, verificados antes de instalarlos
import requests
RELEASE_PESOS_TSS = "r20260730"
SHA256_PESOS_TSS = {
    "Dataset101_TotalSpineSeg_step1_r20260730.zip": "781f81e8c94feccd3cef17d819ba5c293b506e9b1f2b106fc5e31b9dd85887b0",
    "Dataset102_TotalSpineSeg_step2_r20260730.zip": "494c640ab8cb25475bd0fc4338c0931d37079af9dd19403879335c3908c06b46",
}
def sha256_archivo(ruta, bloque=1 << 20):
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for trozo in iter(lambda: f.read(bloque), b""):
            h.update(trozo)
    return h.hexdigest()

os.environ["TOTALSPINESEG_DATA"] = str(DIR_TSS_DATOS)
dir_exports = DIR_TSS_DATOS / "nnUNet" / "exports"
dir_exports.mkdir(parents=True, exist_ok=True)
for nombre, sha_esperado in SHA256_PESOS_TSS.items():
    destino = dir_exports / nombre
    if not destino.exists():
        with requests.get(f"https://github.com/neuropoly/totalspineseg/releases/download/{RELEASE_PESOS_TSS}/{nombre}",
                          stream=True, timeout=120) as r:
            r.raise_for_status()
            with open(destino, "wb") as f:
                for trozo in r.iter_content(1 << 20):
                    f.write(trozo)
    assert sha256_archivo(destino) == sha_esperado, f"{nombre} NO coincide con la huella verificada; no continuar."
subprocess.run([str(VENV_TSS / "bin" / "totalspineseg_init"), "--quiet"], check=True, env=os.environ.copy())
print("Pesos verificados e instalados")

In [ ]:
# 4. Conversión de RSNA con la geometría de las cabeceras, y prueba con una serie
TOL_ORIENTACION = 1e-3                             # diferencia máxima entre las IOP de los cortes de una serie
TOL_PASO = 0.05                                    # irregularidad tolerada del paso entre cortes (fracción)
CSV_GEOMETRIA = TRABAJO / f"geometria_rsna_{LISTA}{FIN}.csv"

def vector(valor):
    return np.array([float(v) for v in valor], dtype=float)

def convertir_con_geometria(carpeta, destino, anotar=True):
    """Lee la serie con GDCM como en el cuaderno 08 y le pone la geometría de las cabeceras DICOM. Devuelve la huella
    del 08 (vóxeles y geometría tal como los leyó GDCM), para comprobar que la entrada es la misma, y la imagen nueva."""
    lector = sitk.ImageSeriesReader()
    ids = lector.GetGDCMSeriesIDs(str(carpeta))
    if not ids:
        raise RuntimeError("GDCM no encontró ninguna serie en la carpeta")
    archivos = list(max((lector.GetGDCMSeriesFileNames(str(carpeta), i) for i in ids), key=len))
    lector.SetFileNames(archivos)
    img_gdcm = lector.Execute()
    huella_08 = conversion08.huella_volumen(img_gdcm)
    arr = sitk.GetArrayFromImage(img_gdcm)          # un corte por archivo, en el orden de `archivos`

    cab = [pydicom.dcmread(a, stop_before_pixels=True, force=True) for a in archivos]
    iops = np.array([vector(c.ImageOrientationPatient) for c in cab])
    if np.abs(iops - iops[0]).max() > TOL_ORIENTACION:
        raise RuntimeError("los cortes no tienen todos la misma orientación")
    fila = iops[0, :3] / np.linalg.norm(iops[0, :3])                 # dirección en que crece la columna (índice i)
    col = iops[0, 3:] - (iops[0, 3:] @ fila) * fila                   # dirección en que crece la fila (índice j)
    col /= np.linalg.norm(col)
    normal = np.cross(fila, col)
    ipp = np.array([vector(c.ImagePositionPatient) for c in cab])
    ps = vector(cab[0].PixelSpacing)                                 # [entre filas, entre columnas]
    if any(not np.allclose(vector(c.PixelSpacing), ps, atol=1e-4) for c in cab):
        raise RuntimeError("los cortes no tienen todos el mismo tamaño de píxel")
    # Orden de los cortes por su posición a lo largo de la normal; el eje de los cortes apunta hacia donde crece
    proyeccion = ipp @ normal
    orden = np.argsort(proyeccion, kind="stable")
    pasos = np.diff(proyeccion[orden])
    if len(archivos) > 1:
        if np.any(pasos < 1e-3):
            raise RuntimeError("hay cortes repetidos en la misma posición")
        paso = float(np.median(pasos))
        irregularidad = float(np.abs(pasos - paso).max() / paso)
    else:
        paso, irregularidad = float(getattr(cab[0], "SpacingBetweenSlices", None) or cab[0].SliceThickness), 0.0
    img = sitk.GetImageFromArray(arr[orden])
    img.SetSpacing((float(ps[1]), float(ps[0]), paso))
    img.SetOrigin(tuple(float(v) for v in ipp[orden[0]]))
    img.SetDirection(tuple(np.column_stack([fila, col, normal]).ravel()))
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    sitk.WriteImage(img, str(destino), useCompression=True)
    if anotar:
        fila_csv = {"id_serie": destino.name.split(".nii")[0], "cortes": len(archivos), "espaciado_i": float(ps[1]),
                    "espaciado_j": float(ps[0]), "paso_cortes": paso, "irregularidad_paso": irregularidad,
                    "paso_irregular": irregularidad > TOL_PASO,
                    "orden_invertido": bool((orden != np.arange(len(orden))).any()),
                    "filas": int(arr.shape[1]), "columnas": int(arr.shape[2]),
                    "espaciado_gdcm": "x".join(f"{v:.3f}" for v in img_gdcm.GetSpacing()),
                    "origen_gdcm_cero": bool(np.allclose(img_gdcm.GetOrigin(), 0)),
                    "huella_corregida": conversion08.huella_volumen(img)}
        pd.DataFrame([fila_csv]).to_csv(CSV_GEOMETRIA, mode="a", header=not CSV_GEOMETRIA.exists(), index=False)
    return huella_08, img

def geometria_deformada(img):
    "La geometría que dejaba la conversión del 08: espaciado 1 × 1 × 1 mm y origen en cero."
    return bool(np.allclose(img.GetSpacing(), 1.0) and np.allclose(img.GetOrigin(), 0.0))

# Prueba con la primera serie de la lista antes de lanzar nada: si la geometría no quedó bien, el cuaderno se detiene
r0 = lista.sort_values("orden").iloc[0]
huella, im = convertir_con_geometria(DIR_RSNA / r0.estudio / r0.serie_clave, TMP / "prueba_geometria.nii.gz", anotar=False)
releida = sitk.ReadImage(str(TMP / "prueba_geometria.nii.gz"))
print("Prueba con una serie:", "huella del 08 igual" if huella == r0.huella_volumen else "huella del 08 DISTINTA",
      "| espaciado", np.round(releida.GetSpacing(), 3), "| origen", np.round(releida.GetOrigin(), 1),
      "| dirección", np.round(releida.GetDirection(), 3))
assert huella == r0.huella_volumen, "Los vóxeles no son los mismos que convirtió el cuaderno 08: no seguir"
assert not geometria_deformada(releida), "La imagen sigue con espaciado 1 × 1 × 1 y origen en cero: no seguir"
assert conversion08.misma_geometria(im, releida), "La geometría cambió al guardar el NIfTI: no seguir"
(TMP / "prueba_geometria.nii.gz").unlink()

In [ ]:
# 5. Segmentación por lotes, en las GPU disponibles a la vez
import torch
N_GPU = max(torch.cuda.device_count(), 1)
CSV_REG = TRABAJO / f"registro_rsna_{LISTA}{FIN}.csv"
ZIP_MASCARAS = TRABAJO / f"mascaras_rsna_{LISTA}{FIN}.zip"
COLUMNAS_REG = ["lista", "orden", "conjunto", "id_serie", "estado", "geometria_ok", "huella_ok", "segundos_lote",
                "series_en_lote", "gpu", "fecha", "error"]

# Series ya segmentadas en versiones anteriores (si se agregó su salida como entrada)
hechas = set()
for reg in buscar(f"registro_rsna_{LISTA}*.csv"):
    r = pd.read_csv(reg, dtype=str)
    hechas |= set(r.loc[r["estado"] == "ok", "id_serie"])
pendientes = lista[(lista["orden"] >= INICIO) & ~lista["id_serie"].isin(hechas)].sort_values("orden")
print(f"GPUs: {N_GPU} | ya hechas en versiones anteriores: {len(hechas)} | "
      f"pendientes desde la posición {INICIO}: {len(pendientes)}")

def preparar_entrada(r, dir_entrada):
    "Deja la serie en NIfTI en la carpeta del lote, con la geometría de las cabeceras."
    huella, _ = convertir_con_geometria(DIR_RSNA / r.estudio / r.serie_clave, dir_entrada / f"{r.id_serie}.nii.gz")
    return huella == r.huella_volumen

def lanzar(lote, gpu):
    "Prepara un lote y lanza TotalSpineSeg en una GPU sin esperar a que termine."
    base = TMP / f"lote_gpu{gpu}"
    shutil.rmtree(base, ignore_errors=True)
    (base / "entrada").mkdir(parents=True)
    huellas = {}
    for r in lote.itertuples():
        try:
            huellas[r.id_serie] = preparar_entrada(r, base / "entrada")
        except Exception as e:
            huellas[r.id_serie] = f"error de entrada: {type(e).__name__}: {e}"[:200]
    cmd = [str(VENV_TSS / "bin" / "totalspineseg"), str(base / "entrada"), str(base / "salida"),
           "--device", "cuda", "--keep-only", "step2_output", "--quiet"]
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "MPLBACKEND": "Agg"}
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, env=env)
    return {"lote": lote, "gpu": gpu, "base": base, "huellas": huellas, "proc": proc, "t0": time.time()}

def mensaje_de_error(texto):
    "Las líneas útiles de la salida de error, sin las barras de progreso."
    lineas = [l.strip() for l in texto.replace("\r", "\n").splitlines()]
    utiles = [l for l in lineas if l and "it/s]" not in l and "s/it]" not in l and "%|" not in l]
    return " | ".join(utiles)[-300:]

def recoger(trabajo):
    "Espera a que termine un lote, guarda sus máscaras y anota cada serie en el registro."
    salida, error = trabajo["proc"].communicate()
    dt = time.time() - trabajo["t0"]
    filas, ok, deformadas = [], 0, 0
    for r in trabajo["lote"].itertuples():
        fila = dict.fromkeys(COLUMNAS_REG, "")
        fila.update({"lista": LISTA, "orden": r.orden, "conjunto": r.conjunto, "id_serie": r.id_serie,
                     "segundos_lote": round(dt, 1), "series_en_lote": len(trabajo["lote"]), "gpu": trabajo["gpu"],
                     "fecha": datetime.datetime.now().isoformat(timespec="seconds")})
        huella = trabajo["huellas"][r.id_serie]
        out = trabajo["base"] / "salida" / "step2_output" / f"{r.id_serie}.nii.gz"
        if isinstance(huella, str):
            fila.update({"estado": "error", "error": huella})
        elif out.exists():
            img = sitk.ReadImage(str(trabajo["base"] / "entrada" / f"{r.id_serie}.nii.gz"))
            seg = sitk.ReadImage(str(out))
            geometria_ok = conversion08.misma_geometria(img, seg)
            if geometria_ok:
                seg.CopyInformation(img)
            deformadas += geometria_deformada(seg)
            local = DIR_MASCARAS / f"{r.id_serie}.nii.gz"
            sitk.WriteImage(sitk.Cast(seg, sitk.sitkUInt8), str(local), useCompression=True)
            with zipfile.ZipFile(ZIP_MASCARAS, "a") as z:       # queda en la salida aunque la sesión se corte después
                z.write(local, local.name)
            local.unlink()
            fila.update({"estado": "ok", "geometria_ok": geometria_ok, "huella_ok": huella})
            ok += 1
        else:
            fila.update({"estado": "error", "error": "TotalSpineSeg sin salida: " + mensaje_de_error(error)})
        filas.append(fila)
    pd.DataFrame(filas, columns=COLUMNAS_REG).to_csv(CSV_REG, mode="a", header=not CSV_REG.exists(), index=False)
    return ok, dt, deformadas

lotes = [pendientes.iloc[i:i + TAM_LOTE] for i in range(0, len(pendientes), TAM_LOTE)]
inicio, siguiente, en_curso, segundos_por_lote = time.time(), 0, [], []
total_ok = 0
while siguiente < len(lotes) or en_curso:
    transcurrido = (time.time() - inicio) / 3600
    margen = (max(segundos_por_lote) if segundos_por_lote else 1800) / 3600
    libres = [g for g in range(N_GPU) if g not in {t["gpu"] for t in en_curso}]
    while libres and siguiente < len(lotes) and transcurrido + margen < HORAS_MAXIMAS:
        en_curso.append(lanzar(lotes[siguiente], libres.pop(0)))
        siguiente += 1
    terminado = None
    while terminado is None and en_curso:
        for t in en_curso:
            if t["proc"].poll() is not None:
                terminado = t
                break
        else:
            time.sleep(5)
    if terminado is None:
        break
    en_curso = [t for t in en_curso if t is not terminado]     # por identidad: comparar los lotes falla
    ok, dt, deformadas = recoger(terminado)
    total_ok += ok
    segundos_por_lote.append(dt)
    print(f"Lote en GPU {terminado['gpu']}: {ok}/{len(terminado['lote'])} ok en {dt:.0f} s "
          f"({dt / len(terminado['lote']):.0f} s por serie) | {total_ok} máscaras | {(time.time() - inicio) / 3600:.2f} h")
    # Los primeros lotes deciden si vale la pena seguir gastando GPU
    if len(segundos_por_lote) <= N_GPU and ok == 0:
        raise RuntimeError("El primer lote no produjo ninguna máscara: revisar el registro antes de seguir")
    if deformadas:
        raise RuntimeError(f"{deformadas} máscaras quedaron con espaciado 1 × 1 × 1 y origen en cero: revisar antes de seguir")

reg = pd.read_csv(CSV_REG, dtype=str) if CSV_REG.exists() else pd.DataFrame(columns=COLUMNAS_REG)
intentadas = set(reg["id_serie"]) | hechas
faltan = lista[~lista["id_serie"].isin(intentadas)].sort_values("orden")
print(f"\nEsta versión: {total_ok} máscaras | errores: {(reg['estado'] == 'error').sum()} | "
      f"huellas distintas: {(reg['huella_ok'] == 'False').sum()} | geometría distinta: {(reg['geometria_ok'] == 'False').sum()}")
print("Siguiente INICIO:", int(faltan["orden"].min()) if len(faltan) else "ninguno, todas las series de la lista se intentaron")

In [ ]:
# 6. Comprobación final y registro del entorno
from importlib.metadata import version as v
with zipfile.ZipFile(ZIP_MASCARAS) as z:
    en_zip = {n.split(".nii")[0] for n in z.namelist()}
ok_registro = set(reg.loc[reg["estado"] == "ok", "id_serie"])
print(f"Máscaras en {ZIP_MASCARAS.name}: {len(en_zip)} ({ZIP_MASCARAS.stat().st_size / 1e6:.1f} MB) | "
      f"correctas en el registro: {len(ok_registro)} | sin máscara en el zip: {len(ok_registro - en_zip)}")
geo = pd.read_csv(CSV_GEOMETRIA)
print(f"Geometría: {len(geo)} series convertidas | GDCM con espaciado 1 mm: {(geo['espaciado_gdcm'] == '1.000x1.000x1.000').mean():.0%} | "
      f"paso irregular: {int(geo['paso_irregular'].sum())} | espaciado en el plano (mediana): {geo['espaciado_i'].median():.3f} mm | "
      f"paso entre cortes (mediana): {geo['paso_cortes'].median():.2f} mm")
errores = reg[reg["estado"] == "error"]["error"].str.replace(r"rsna_\d+_\d+", "<serie>", regex=True)
print("Errores más frecuentes (sin identificadores):")
print(errores.str[:150].value_counts().head(5).to_string() if len(errores) else "ninguno")
entorno = {"fecha": datetime.datetime.now().isoformat(timespec="seconds"), "lista": LISTA, "inicio": INICIO,
           "corrida": "RSNA con la geometría de las cabeceras DICOM",
           "python": sys.version.split()[0], "gpus": N_GPU,
           "gpu_nombre": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
           "totalspineseg": "20260807", "pesos_release": RELEASE_PESOS_TSS, "pesos_sha256": SHA256_PESOS_TSS,
           "tam_lote": TAM_LOTE, "simpleitk": v("SimpleITK"), "pydicom": v("pydicom"),
           "conversion08_sha256": sha256_archivo(RUTA_CONVERSION), "tol_orientacion": TOL_ORIENTACION, "tol_paso": TOL_PASO}
(TRABAJO / f"entorno_rsna_{LISTA}{FIN}.json").write_text(json.dumps(entorno, indent=2, ensure_ascii=False))
print(json.dumps(entorno, indent=2, ensure_ascii=False))